# Avlokan · Phase 1 — Indore's water bodies

*Divya Drishti for every city.*

This notebook measures how much open water Indore's lakes hold, date by date, from Sentinel-2 imagery since 2017, and whether they're shrinking.

1. **Part A** runs the whole pipeline on synthetic data (offline, instant) so you can see what the outputs look like.
2. **Part B** runs it on real imagery from Microsoft Planetary Computer.
3. **Part C** is a playground: compare indices and thresholds.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # so `import avlokan` works from notebooks/

from IPython.display import Image, display
from avlokan.config import LAKES
from avlokan.pipeline import run_from_stack, run_lake

{k: (v.name, v.bbox, "verified" if v.verified else "NOT verified") for k, v in LAKES.items()}

## Part A — dry run on synthetic data
A fake circular lake that fills each monsoon and loses 4% a year, with random clouds.

In [ ]:
from avlokan.data.synthetic import synthetic_stack

report = run_from_stack(LAKES["sirpur"], synthetic_stack(), "../outputs/_synthetic")
print(json.dumps(report, indent=2))
display(Image("../outputs/_synthetic/sirpur/water_timeseries.png"))
display(Image("../outputs/_synthetic/sirpur/before_after.png"))

## Part B — real Sentinel-2 imagery

**First, check the lake box on a map.** Paste the GeoJSON below into [geojson.io](https://geojson.io). The box should contain the whole lake with some land around it. If it's off, fix the coordinates in `avlokan/config.py` and set `verified=True`.

In [ ]:
LAKE = "sirpur"   # sirpur | bilawali | yeshwant_sagar | pipliyapala
aoi = LAKES[LAKE]
x0, y0, x1, y1 = aoi.bbox
print(json.dumps({"type": "Polygon", "coordinates": [[[x0, y0], [x1, y0], [x1, y1], [x0, y1], [x0, y0]]]}))

Start with a short window (one or two years) to check things work, then widen to `2017-01-01`. Downloads only the small cut-out around the lake, so it's light on bandwidth.

In [ ]:
from avlokan.data.stac import search_scenes, dedupe_by_day, load_stack

items = dedupe_by_day(search_scenes(aoi, "2023-01-01", "2024-12-31"))
print(len(items), "scenes, first:", items[0].datetime.date(), "last:", items[-1].datetime.date())

In [ ]:
stack = load_stack(items, aoi)
report = run_from_stack(aoi, stack, "../outputs")
print(json.dumps(report, indent=2))
out = Path("../outputs") / LAKE
display(Image(out / "water_timeseries.png"))
display(Image(out / "before_after.png"))

In [ ]:
import pandas as pd
ts = pd.read_csv(out / "water_timeseries.csv", parse_dates=["date"])
print(f"{ts['usable'].sum()} usable of {len(ts)} dates")
pd.read_csv(out / "yearly_summary.csv")

## Part C — playground

- **MNDWI vs NDWI**: MNDWI uses SWIR and usually handles urban edges better; NDWI uses NIR.
- **Fixed vs Otsu**: fixed `0.0` is the textbook threshold; Otsu adapts per image (useful with algae or turbid water).

Look at where they disagree; those disagreements are exactly what the Phase 2 ML model should learn to resolve.

In [ ]:
from avlokan.analysis.timeseries import water_timeseries

variants = {
    "mndwi / fixed": water_timeseries(stack, index="mndwi", method="fixed"),
    "mndwi / otsu":  water_timeseries(stack, index="mndwi", method="otsu"),
    "ndwi / fixed":  water_timeseries(stack, index="ndwi",  method="fixed"),
}
pd.DataFrame({k: v[v["usable"]].set_index("date")["water_ha"] for k, v in variants.items()}).plot(
    figsize=(11, 4), marker=".", lw=0.8, title=f"{aoi.name}: method comparison (ha)")

## Run every lake from the terminal

```bash
python scripts/run_water_timeseries.py --lake all
```